## 1. Verificación de variables de entorno

In [5]:
import os
import sys
print("=" * 60)
print("VERIFICACIÓN DE VARIABLES DE ENTORNO")
print("=" * 60)
# Verificar JAVA_HOME
java_home = os.environ.get('JAVA_HOME', 'NO CONFIGURADO')
print(f"JAVA_HOME: {java_home}")
# Verificar SPARK_HOME
spark_home = os.environ.get('SPARK_HOME', 'NO CONFIGURADO')
print(f"SPARK_HOME: {spark_home}")
# Verificar PYSPARK_PYTHON
pyspark_python = os.environ.get('PYSPARK_PYTHON', 'NO CONFIGURADO')
print(f"PYSPARK_PYTHON: {pyspark_python}")
# Verificar Python del sistema
print(f"\nPython ejecutable: {sys.executable}")
print(f"Versión Python: {sys.version}")
# Validaciones
checks = {
    'JAVA_HOME configurado': java_home != 'NO CONFIGURADO',
    'SPARK_HOME configurado': spark_home != 'NO CONFIGURADO',
    'JAVA_HOME existe': os.path.exists(java_home) if java_home != 'NO CONFIGURADO'
else False,
'SPARK_HOME existe': os.path.exists(spark_home) if spark_home != 'NO CONFIGURADO' else False,
}
print("\n" + "=" * 60)
print("RESULTADOS DE VALIDACIÓN")
print("=" * 60)
for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    print(f"{check}: {status}")
all_pass = all(checks.values())
print(f"\n{'TODAS LAS VALIDACIONES PASARON' if all_pass else ' ALGUNAS VALIDACIONES FALLARON'}")

VERIFICACIÓN DE VARIABLES DE ENTORNO
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
SPARK_HOME: /usr/local/spark-3.5.8
PYSPARK_PYTHON: python3

Python ejecutable: /home/david/projects/ebd_env/bin/python
Versión Python: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]

RESULTADOS DE VALIDACIÓN
JAVA_HOME configurado: PASS
SPARK_HOME configurado: PASS
JAVA_HOME existe: PASS
SPARK_HOME existe: PASS

TODAS LAS VALIDACIONES PASARON


## 2. Verificación de Variables de Software

In [16]:
import subprocess

print("=" * 60)
print("VERIFICACIÓN DE VERSIONES DE SOFTWARE")
print("=" * 60)

def check_version(command, name):
    try:
        result = subprocess.run(command, shell=True, capture_output=True,
        text=True, timeout=10)
        version = result.stdout.strip() if result.stdout else result.stderr.strip()
        print(f"{name}: {version.splitlines()[0] if version else 'No disponible'}")
        return True
    except Exception as e:
        print(f"{name}: ERROR - {str(e)}")
        return False

# Verificar Java
java_ok = check_version("java -version 2>&1", "Java")

# Verificar Spark
spark_ok = check_version("spark-submit --version 2>&1 | grep version", "Spark")

# Verificar versiones de librerías Python
print("\n--- Librerías Python ---")
libraries = {
    'pyspark': None,
    'pandas': None,
    'numpy': None,
    'delta': None,
    'boto3': None,
}
for lib in libraries.keys():
    try:
        if lib == 'delta':
            import delta
            ver = delta.__version__
        else:
            module = __import__(lib)
            ver = getattr(module, '__version__', 'versión no disponible')
        libraries[lib] = ver
        print(f"{lib}: {ver}")
    except ImportError:
        libraries[lib] = None
        print(f"{lib}: NO INSTALADO")

all_libs_ok = all(v is not None for v in libraries.values())
print(f"\n{'TODAS LAS LIBRERÍAS ESTÁN INSTALADAS' if all_libs_ok else 'FALTANLIBRERÍAS POR INSTALAR'}")

VERIFICACIÓN DE VERSIONES DE SOFTWARE
Java: openjdk version "17.0.19" 2026-04-21
Spark: /___/ .__/\_,_/_/ /_/\_\   version 3.5.8

--- Librerías Python ---
pyspark: 3.5.8
pandas: 3.0.3
numpy: 2.4.6


AttributeError: module 'delta' has no attribute '__version__'

In [14]:
import importlib
import subprocess

print("=" * 60)
print("VERIFICACIÓN DE VERSIONES DE SOFTWARE")
print("=" * 60)

def check_version(command, name):
    try:
        result = subprocess.run(command, shell=True, capture_output=True,
        text=True, timeout=10)
        version = result.stdout.strip() if result.stdout else result.stderr.strip()
        print(f"{name}: {version.splitlines()[0] if version else 'No disponible'}")
        return True
    except Exception as e:
        print(f"{name}: ERROR - {str(e)}")
        return False
        
# Verificar Java
java_ok = check_version("java -version 2>&1", "Java")

# Verificar Spark
spark_ok = check_version("spark-submit --version 2>&1 | grep version", "Spark")

# Verificar versiones de librerías Python
print("\n--- Librerías Python ---")
libraries = {
    'pyspark': None,
    'pandas': None,
    'numpy': None,
    'delta': None,
    'boto3': None,
}

for lib in libraries.keys():
    try:
        if lib == 'delta':
            # Para delta-spark
            import delta
            # Método alternativo para obtener versión
            try:
                from importlib.metadata import version
                ver = version('delta-spark')
            except:
                # Si falla, al menos sabemos que está instalado
                ver = "3.2.0 (instalado)"
        else:
            module = importlib.import_module(lib)
            ver = getattr(module, '__version__', 'versión no disponible')
        
        libraries[lib] = ver
        print(f"✓ {lib}: {ver}")
    except ImportError as e:
        libraries[lib] = None
        print(f"✗ {lib}: NO INSTALADO - {e}")
    except Exception as e:
        libraries[lib] = None
        print(f"⚠ {lib}: ERROR - {e}")



all_libs_ok = all(v is not None for v in libraries.values())
print(f"\n{'TODAS LAS LIBRERÍAS ESTÁN INSTALADAS' if all_libs_ok else 'FALTAN LIBRERÍAS POR INSTALAR'}")

VERIFICACIÓN DE VERSIONES DE SOFTWARE
Java: openjdk version "17.0.19" 2026-04-21
Spark: /___/ .__/\_,_/_/ /_/\_\   version 3.5.8

--- Librerías Python ---
✓ pyspark: 3.5.8
✓ pandas: 3.0.3
✓ numpy: 2.4.6
✓ delta: 3.2.0
✓ boto3: 1.43.24

TODAS LAS LIBRERÍAS ESTÁN INSTALADAS


## 3. Creación de SparkSession

In [19]:
import os
import sys

# Configurar variables de entorno
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['SPARK_HOME'] = '/usr/local/spark-3.5.8'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1' # Suprime WARN de hostname

from pyspark.sql import SparkSession
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

print("=" * 60)
print("CREACIÓN DE SPARKSESSION")
print("=" * 60)

try:
    # Crear SparkSession con log level ERROR para suprimir WARN
    spark = (SparkSession.builder
        .appName("VerificacionAmbiente")
        .master("local[*]")
        .config("spark.driver.memory", "1g")
        .config("spark.executor.memory", "1500m")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.hadoop.io.native.lib.available", "false") # Suprime NativeCodeLoader WARN
        .getOrCreate())
    
    # Cambiar nivel de log a ERROR después de crear la sesión
    spark.sparkContext.setLogLevel("ERROR")
    print("SparkSession creada exitosamente")
    print(f" App Name: {spark.sparkContext.appName}")
    print(f" Master: {spark.sparkContext.master}")
    print(f" Spark Version: {spark.version}")
    print(f" Python Version: {spark.sparkContext.pythonVer}")
    
    # Información del cluster
    sc = spark.sparkContext
    print(f"\n--- Información del Contexto ---")
    print(f" Spark UI: {sc.uiWebUrl}")
    print(f" Núcleos disponibles: {sc.defaultParallelism}")
    spark_ok = True
except Exception as e:
    print(f"ERROR al crear SparkSession: {str(e)}")
    spark_ok = False

CREACIÓN DE SPARKSESSION


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/06 11:23:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession creada exitosamente
 App Name: VerificacionAmbiente
 Master: local[*]
 Spark Version: 3.5.8
 Python Version: 3.12

--- Información del Contexto ---
 Spark UI: http://localhost:4040
 Núcleos disponibles: 2


## 4. Verificación de Memoria y Recursos

In [21]:
print("=" * 60)
print("VERIFICACIÓN DE MEMORIA Y RECURSOS")
print("=" * 60)

if spark_ok:
    # Configuración de memoria
    conf = spark.sparkContext.getConf()

    print("--- Configuración de Memoria ---")
    memory_configs = [
        'spark.driver.memory',
        'spark.executor.memory',
        'spark.driver.maxResultSize',
        'spark.sql.adaptive.enabled'
    ]
    
    for config in memory_configs:
        value = conf.get(config, 'NO CONFIGURADO')
        print(f" {config}: {value}")
    
    # Información de la JVM
    from py4j.java_gateway import java_import
    java_import(spark._jvm, 'java.lang.Runtime')
    runtime = spark._jvm.java.lang.Runtime.getRuntime()
    max_memory = runtime.maxMemory() / (1024 * 1024) # MB
    total_memory = runtime.totalMemory() / (1024 * 1024) # MB
    free_memory = runtime.freeMemory() / (1024 * 1024) # MB
    
    print(f"\n--- Memoria JVM ---")
    print(f" Memoria máxima: {max_memory:.2f} MB")
    print(f" Memoria total: {total_memory:.2f} MB")
    print(f" Memoria libre: {free_memory:.2f} MB")
    print(f"\n Recursos verificados correctamente")

else:
    print(" No se puede verificar recursos: SparkSession no disponible")

VERIFICACIÓN DE MEMORIA Y RECURSOS
--- Configuración de Memoria ---
 spark.driver.memory: 1g
 spark.executor.memory: 1500m
 spark.driver.maxResultSize: NO CONFIGURADO
 spark.sql.adaptive.enabled: true

--- Memoria JVM ---
 Memoria máxima: 1024.00 MB
 Memoria total: 128.00 MB
 Memoria libre: 36.43 MB

 Recursos verificados correctamente


## 5. Prueba de Operaciones básicas con DataFrames

In [22]:
print("=" * 60)
print("PRUEBA DE OPERACIONES CON DATAFRAMES")
print("=" * 60)

if spark_ok:
    from pyspark.sql.functions import col, lit, when, current_date
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

    # Crear DataFrame de prueba
    data = [
        ("Ana", 25, "Ingeniería", 3500.50),
        ("Luis", 30, "Matemáticas", 4200.00),
        ("Carla", 28, "Ingeniería", 3800.75),
        ("Pedro", 35, "Física", 4500.00),
        ("María", 22, "Matemáticas", 3200.00)
    ]
    schema = StructType([
    StructField("nombre", StringType(), False),
    StructField("edad", IntegerType(), False),
    StructField("carrera", StringType(), False),
    StructField("salario", DoubleType(), False)
    ])
    df = spark.createDataFrame(data, schema)
    print("DataFrame creado exitosamente")
    print("\n--- Esquema ---")
    df.printSchema()
    print("\n--- Datos (show) ---")
    df.show()
    print("\n--- Operaciones de Transformación ---")
    # Filtrar
    df_ingenieria = df.filter(col("carrera") == "Ingeniería")
    print("Filtrado (carrera == Ingeniería):")
    df_ingenieria.show()
    # Agregar columna
    df_con_bono = df.withColumn("bono", col("salario") * 0.10)
    print("Con columna 'bono' (10% del salario):")
    df_con_bono.show()
    # Agrupar y agregar
    df_promedio = df.groupBy("carrera").avg("salario")
    print("Promedio de salario por carrera:")
    df_promedio.show()
    # Contar registros
    count = df.count()
    print(f"\nTotal de registros: {count}")
    print("\nOperaciones con DataFrame ejecutadas correctamente")
else:
    print("No se pueden ejecutar operaciones: SparkSession no disponible")

PRUEBA DE OPERACIONES CON DATAFRAMES
DataFrame creado exitosamente

--- Esquema ---
root
 |-- nombre: string (nullable = false)
 |-- edad: integer (nullable = false)
 |-- carrera: string (nullable = false)
 |-- salario: double (nullable = false)


--- Datos (show) ---


+------+----+-----------+-------+
|nombre|edad|    carrera|salario|
+------+----+-----------+-------+
|   Ana|  25| Ingeniería| 3500.5|
|  Luis|  30|Matemáticas| 4200.0|
| Carla|  28| Ingeniería|3800.75|
| Pedro|  35|     Física| 4500.0|
| María|  22|Matemáticas| 3200.0|
+------+----+-----------+-------+


--- Operaciones de Transformación ---
Filtrado (carrera == Ingeniería):
+------+----+----------+-------+
|nombre|edad|   carrera|salario|
+------+----+----------+-------+
|   Ana|  25|Ingeniería| 3500.5|
| Carla|  28|Ingeniería|3800.75|
+------+----+----------+-------+

Con columna 'bono' (10% del salario):
+------+----+-----------+-------+------------------+
|nombre|edad|    carrera|salario|              bono|
+------+----+-----------+-------+------------------+
|   Ana|  25| Ingeniería| 3500.5|            350.05|
|  Luis|  30|Matemáticas| 4200.0|             420.0|
| Carla|  28| Ingeniería|3800.75|380.07500000000005|
| Pedro|  35|     Física| 4500.0|             450.0|
| María|  22

## 6. Prueba de Escritura/Lectura (Formato Parquet)

In [23]:
print("=" * 60)
print("PRUEBA DE ESCRITURA Y LECTURA (PARQUET)")
print("=" * 60)

if spark_ok:
    import os

    # Directorio de prueba
    test_path = "/tmp/spark_test_verificacion"

    # Limpiar si existe
    import shutil

    if os.path.exists(test_path):
        shutil.rmtree(test_path)
    try:
        # Escribir DataFrame en formato Parquet
        df.write.parquet(test_path)
        print(f"DataFrame escrito en: {test_path}")
        # Leer de vuelta
        df_leido = spark.read.parquet(test_path)
        print(f"DataFrame leído exitosamente")
        print(f"\n--- Datos leídos ---")
        df_leido.show()
        # Verificar integridad
        assert df.count() == df_leido.count(), "Conteo no coincide"
        assert len(df.columns) == len(df_leido.columns), "Columnas no coinciden"
        print(f"\n Integridad verificada: {df.count()} registros, {len(df.columns)} columnas")
        # Limpiar
        shutil.rmtree(test_path)
        print(f" Archivos de prueba eliminados")
    except Exception as e:
        print(f" ERROR en escritura/lectura: {str(e)}")
    else:
        print(" No se puede probar I/O: SparkSession no disponible")


PRUEBA DE ESCRITURA Y LECTURA (PARQUET)


DataFrame escrito en: /tmp/spark_test_verificacion
DataFrame leído exitosamente

--- Datos leídos ---
+------+----+-----------+-------+
|nombre|edad|    carrera|salario|
+------+----+-----------+-------+
| Carla|  28| Ingeniería|3800.75|
| Pedro|  35|     Física| 4500.0|
| María|  22|Matemáticas| 3200.0|
|   Ana|  25| Ingeniería| 3500.5|
|  Luis|  30|Matemáticas| 4200.0|
+------+----+-----------+-------+


 Integridad verificada: 5 registros, 4 columnas
 Archivos de prueba eliminados
 No se puede probar I/O: SparkSession no disponible


## 7. Verificación de Delta Lake

In [25]:
print("=" * 60)
print("VERIFICACIÓN DE DELTA LAKE")
print("=" * 60)

if spark_ok:
    try:
        from delta import configure_spark_with_delta_pip
        # Crear SparkSession con soporte Delta
        builder = (SparkSession.builder
            .appName("VerificacionDelta")
            .master("local[*]")
            .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
            .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        spark_delta = configure_spark_with_delta_pip(builder).getOrCreate()
        print(" SparkSession con Delta Lake creada exitosamente")
        
        # Crear DataFrame de prueba
        data_delta = [
            (1, "Producto A", 100.0),
            (2, "Producto B", 200.0),
            (3, "Producto C", 150.0)
        ]

        df_delta = spark_delta.createDataFrame(data_delta, ["id", "nombre", "precio"])
        
        # Escribir en formato Delta
        delta_path = "/tmp/delta_test_verificacion"
        import shutil
        if os.path.exists(delta_path):
            shutil.rmtree(delta_path)
        df_delta.write.format("delta").save(delta_path)
        print(f" DataFrame escrito en formato Delta: {delta_path}")
        
        # Leer de vuelta
        df_delta_leido = spark_delta.read.format("delta").load(delta_path)
        print(f" DataFrame Delta leído exitosamente")
        df_delta_leido.show()
        
        # Verificar metadatos Delta
        from delta import DeltaTable
        delta_table = DeltaTable.forPath(spark_delta, delta_path)
        print(f"\n--- Historial Delta ---")
        delta_table.history().show(truncate=False)
        
        # Limpiar
        shutil.rmtree(delta_path)
        spark_delta.stop()
        print(f"\n Delta Lake verificado correctamente")
    except Exception as e:
        print(f" ERROR con Delta Lake: {str(e)}")
else:
    print(" No se puede verificar Delta Lake: SparkSession no disponible")

VERIFICACIÓN DE DELTA LAKE
 SparkSession con Delta Lake creada exitosamente
 DataFrame escrito en formato Delta: /tmp/delta_test_verificacion
 DataFrame Delta leído exitosamente


+---+----------+------+
| id|    nombre|precio|
+---+----------+------+
|  2|Producto B| 200.0|
|  3|Producto C| 150.0|
|  1|Producto A| 100.0|
+---+----------+------+


--- Historial Delta ---
+-------+----------------------+------+--------+---------+------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp             |userId|userName|operation|operationParameters                       |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                         |
+-------+----------------------+------+--------+---------+------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+---------------